# Day 14 结构特征候选方案 official test 最终观察

Day 14 只对 Day 13 valid 选出的少数候选方案做 official test 最终观察。本轮不在 test 上重新选择策略、结构特征或阈值。

## 1. 关键边界

- 候选方案来自 Day 13 valid 结果；
- 阈值使用 Day 13 valid best threshold；
- imputer 和结构特征 builder 只在 train_inner 上 fit；
- official test 只 transform 并做最终观察；
- test 结果不能反向修改候选方案或阈值；
- `structural_all` 只是上限观察，不直接作为最终主方案；
- 不解释匿名字段真实物理含义。

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd

from scania_aps.config import get_config
from scania_aps.data.load_data import load_train_test_with_target
from scania_aps.data.split_data import split_train_valid
from scania_aps.features.structural_feature_design import load_structural_feature_config
from scania_aps.models.structural_feature_test_evaluation import (
    DEFAULT_DAY14_CANDIDATE_GROUPS,
    evaluate_structural_feature_candidates_on_test,
)

cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")
structural_config = load_structural_feature_config(PROJECT_ROOT / "config" / "structural_features.yaml")
valid_best_summary = pd.read_csv(cfg.metrics_dir / "day13_structural_feature_valid_best_summary.csv")

## 2. 固定候选方案

候选组固定为 baseline、selected missing indicators、prefix zero rate 和 structural_all。这里不会根据 test 表现再增删候选。

In [2]:
candidate_groups = DEFAULT_DAY14_CANDIDATE_GROUPS
valid_best_summary[valid_best_summary["strategy"].isin(candidate_groups)][[
    "strategy", "best_threshold", "precision", "recall", "f2", "average_precision", "fp", "fn", "total_cost"
]]

,strategy,best_threshold,precision,recall,f2,average_precision,fp,fn,total_cost
0,median_all_structural_all,0.18,0.367925,0.975,0.733083,0.863543,335,5,5850
1,median_all_selected_missing_indicators_top30,0.30,0.430804,0.965,0.773237,0.860375,255,7,6050
2,median_all_prefix_zero_rate,0.09,0.293592,0.985,0.669613,0.862527,474,3,6240
4,baseline_median_all,0.16,0.359259,0.970,0.723881,0.867239,346,6,6460


## 3. official test 最终观察

读取 official train/test 后，仅从 official train 内部划分 `train_inner / valid`。模型训练仍然只基于 train_inner，test 只做 transform 和最终观察。

In [3]:
train_df, test_df = load_train_test_with_target(cfg)
train_inner_df, valid_df = split_train_valid(train_df, cfg)

results = evaluate_structural_feature_candidates_on_test(
    train_inner_df=train_inner_df,
    valid_df=valid_df,
    test_df=test_df,
    cfg=cfg,
    structural_config=structural_config,
    candidate_groups=candidate_groups,
    valid_best_summary=valid_best_summary,
)

test_results = results["test_results"]
test_predictions = results["test_predictions"]
valid_test_compare = results["test_compare_with_valid"]
metadata = results["metadata"]

## 4. test 结果表

重点看 total cost、FN、recall、F2 和 AP，不使用 accuracy 作为核心指标。

In [4]:
display_cols = [
    "candidate_group", "threshold", "precision", "recall", "f2",
    "average_precision", "fp", "fn", "total_cost", "n_structural_features"
]
test_results[display_cols]

,candidate_group,threshold,precision,recall,f2,average_precision,fp,fn,total_cost,n_structural_features
0,median_all_structural_all,0.18,0.477004,0.968000,0.802742,0.905455,398,12,9980,60
1,baseline_median_all,0.16,0.455919,0.965333,0.789015,0.908623,432,13,10820,0
2,median_all_prefix_zero_rate,0.09,0.384615,0.973333,0.745202,0.909259,584,10,10840,5
3,median_all_selected_missing_indicators_top30,0.30,0.545736,0.938667,0.820513,0.908459,293,23,14430,30


## 5. valid vs test 对比

这张表用于观察 valid 选择结果是否泛化。即使 test 上某个方案更好，也不能回头修改 Day 13 的选择逻辑。

In [5]:
valid_test_compare

,candidate_group,threshold,valid_precision,valid_recall,valid_f1,valid_f2,valid_average_precision,valid_fp,valid_fn,valid_total_cost,...,test_average_precision,test_fp,test_fn,test_total_cost,delta_total_cost_test_minus_valid,delta_fn_test_minus_valid,delta_fp_test_minus_valid,delta_recall_test_minus_valid,delta_f2_test_minus_valid,note
0,median_all_structural_all,0.18,0.367925,0.975,0.534247,0.733083,0.863543,335,5,5850,...,0.905455,398,12,9980,4130,7,63,-0.007000,0.069659,test 仅用于最终观察；不允许根据该表反向修改候选组或阈值
1,baseline_median_all,0.16,0.359259,0.970,0.524324,0.723881,0.867239,346,6,6460,...,0.908623,432,13,10820,4360,7,86,-0.004667,0.065134,test 仅用于最终观察；不允许根据该表反向修改候选组或阈值
2,median_all_prefix_zero_rate,0.09,0.293592,0.985,0.452354,0.669613,0.862527,474,3,6240,...,0.909259,584,10,10840,4600,7,110,-0.011667,0.075590,test 仅用于最终观察；不允许根据该表反向修改候选组或阈值
3,median_all_selected_missing_indicators_top30,0.30,0.430804,0.965,0.595679,0.773237,0.860375,255,7,6050,...,0.908459,293,23,14430,8380,16,38,-0.026333,0.047276,test 仅用于最终观察；不允许根据该表反向修改候选组或阈值


## 6. 保存输出

输出文件与 Day 13 分开命名，不覆盖旧结果。

In [6]:
test_results.to_csv(cfg.metrics_dir / "day14_structural_feature_test_results.csv", index=False)
valid_test_compare.to_csv(cfg.metrics_dir / "day14_structural_feature_valid_test_compare.csv", index=False)
test_predictions.to_csv(cfg.predictions_dir / "day14_structural_feature_test_predictions.csv", index=False)
metadata.to_csv(cfg.tables_dir / "day14_structural_feature_test_metadata.csv", index=False)

## 7. Day 14 小结

本轮结论需要区分：

1. 哪些结构特征在 valid 上有效，到了 official test 仍然保持较低成本；
2. 哪些结构特征 valid 提升明显，但 test 上泛化不足；
3. `structural_all` 是否只是上限观察，是否存在冗余风险；
4. `selected_missing_indicators_top30` 是否更适合作为主候选；
5. 下一步是继续做轻量调参、结构特征筛选，还是进入模型解释性分析。